## CCA-F Lab 03: Arquitectura Multi-Agente Hub-and-Spoke (Día 5)

### Objetivos de Aprendizaje para el Examen:
1. **Responsabilidades del Hub (Coordinador):** Descomposición del problema original, delegación táctica, agregación de reportes finales y manejo de fallas de los subagentes.
2. **Responsabilidades del Spoke (Subagente):** Ejecución ultra-enfocada en un dominio específico utilizando herramientas acotadas (*scoped permissions*).
3. **Context Isolation (Aislamiento Extremo):** Los subagentes no comparten el array `messages` del coordinador. El Hub debe transferir la información estrictamente necesaria de forma serializada. Esto evita la saturación de tokens y disminuye drásticamente las alucinaciones.
4. **Task Gaps:** Si un subagente entrega un resultado incompleto, la directriz de Anthropic es **corregir la lógica de descomposición en el Hub**, no sobrecargar al subagente con más responsabilidades.

In [1]:
import os
import json
from anthropic import Anthropic
from dotenv import load_dotenv

# Cargar variables de entorno desde la raíz
load_dotenv(dotenv_path="../.env")
client = Anthropic()
print("✅ Entorno multi-agente listo.")

✅ Entorno multi-agente listo.


In [10]:
# Herramienta operativa ultra-específica (Solo accesible por el Spoke de finanzas)
def calculate_sla_breach_penalty(tier: str, hours_over_sla: int) -> dict:
    """Calcula penalizaciones contractuales por violaciones de SLA corporativos."""
    rates = {"ENTERPRISE": 500, "GOLD": 200, "SILVER": 50}
    hourly_rate = rates.get(tier.upper(), 10)
    total_penalty = hourly_rate * hours_over_sla
    return {
        "tier_evaluated": tier,
        "hours_over_sla": hours_over_sla,
        "hourly_penalty_usd": hourly_rate,
        "total_contractual_penalty_usd": total_penalty
    }

# Lógica del subagente especialista (Worker / Spoke)
def run_financial_subagent(target_customer_data: str, incident_duration_hours: int) -> str:
    print("🤖 [Spoke Agent] Ejecutando Subagente de Impacto Financiero con contexto aislado...")
    
    # Declaramos la herramienta en el catálogo del subagente
    spoke_tools = [{
        "name": "calculate_sla_breach_penalty",
        "description": "Calculates the dynamic financial impact penalty based on customer tier classification and total hours exceeding SLA limits.",
        "input_schema": {
            "type": "object",
            "properties": {
                "tier": {"type": "string", "description": "The customer subscription tier (e.g., ENTERPRISE, GOLD)."},
                "hours_over_sla": {"type": "integer"}
            },
            "required": ["tier", "hours_over_sla"]
        }
    }]
    
    # Inyectamos de manera explícita la información dentro de etiquetas XML puras
    spoke_prompt = f"""
    Analyze the following financial exposure:
    <customer_data>{target_customer_data}</customer_data>
    <incident_duration_hours>{incident_duration_hours}</incident_duration_hours>
    
    Calculate the precise penalties using your calculation tool and provide a brief financial summary.
    """
    
    # El bucle interno canónico del subagente (Bucle simplificado para el ejercicio)
    messages = [{"role": "user", "content": spoke_prompt}]
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system="You are a senior financial risk sub-agent. Use your calculation tool immediately to audit numbers.",
        tools=spoke_tools,
        messages=messages
    )
    
    # Si requiere usar la herramienta, la ejecutamos en su entorno
    if response.stop_reason == "tool_use":
        block = [b for b in response.content if b.type == "tool_use"][0]
        if block.name == "calculate_sla_breach_penalty":
            tool_result = calculate_sla_breach_penalty(**block.input)
            
            # Seguir el ciclo agéntico enviando de vuelta el resultado
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [{"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(tool_result)}]
            })
            
            final_subagent_resp = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1000,
                system="You are a senior financial risk sub-agent.",
                messages=messages
            )
            return final_subagent_resp.content[0].text
            
    return response.content[0].text

In [11]:
# Esta es la herramienta que el Hub utilizará para orquestar (El "Task tool" conceptual)
def delegate_financial_audit(customer_json_str: str, outage_hours: int) -> dict:
    """Herramienta de delegación que levanta el subagente financiero con contexto limpio."""
    print(f"🛰️ [Hub Orchestrator] Delegando auditoría financiera a subagente...")
    
    # Llamamos a la función aislada del subagente
    subagent_report = run_financial_subagent(customer_json_str, outage_hours)
    
    return {
        "status": "DELEGATION_COMPLETED",
        "subagent_report_output": subagent_report
    }

# Catálogo de herramientas que conoce el Coordinador Central (Hub)
HUB_TOOLS = [
    {
        "name": "delegate_financial_audit",
        "description": "Spawns a specialized isolated financial compliance sub-agent to audit contractual breach risks and compute penalty fees.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_json_str": {"type": "string", "description": "Raw string containing customer metadata fields."},
                "outage_hours": {"type": "integer", "description": "Total consecutive down-time hours recorded."}
            },
            "required": ["customer_json_str", "outage_hours"]
        }
    }
]

def run_hub_orchestrator(complex_user_prompt: str):
    print(f"[🔥 HUB START] Solicitud Compleja: {complex_user_prompt}\n")
    messages = [{"role": "user", "content": complex_user_prompt}]
    
    # LLamada al Hub Orquestador
    response = client.messages.create(
        model="claude-sonnet-4-6", # Usamos Claude Sonnet 4.6 para esta prueba
        max_tokens=1500,
        system="You are the Lead Master Infrastructure Coordinator. Your task is to delegate sub-analysis using specialized tools.",
        tools=HUB_TOOLS,
        messages=messages
    )
    
    messages.append({"role": "assistant", "content": response.content})
    
    if response.stop_reason == "tool_use":
        block = [b for b in response.content if b.type == "tool_use"][0]
        print(f"🎯 Hub decide delegar usando la herramienta: '{block.name}'")
        
        if block.name == "delegate_financial_audit":
            # Ejecutar la delegación (Backend activa Spoke)
            result = delegate_financial_audit(**block.input)
            
            # Devolver el reporte consolidado del subagente al Hub
            messages.append({
                "role": "user",
                "content": [{"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(result)}]
            })
            
            # Turno de cierre del Hub para dar la síntesis ejecutiva al usuario final
            final_hub_response = client.messages.create(
                model="claude-sonnet-4-6", # Usamos Claude Sonnet 4.6 para esta prueba
                max_tokens=1500,
                system="You are the Lead Master Infrastructure Coordinator.",
                messages=messages
            )
            
            print("\n👑 [RESPUESTA FINAL SINTETIZADA POR EL HUB]:")
            print(final_hub_response.content[0].text)

In [12]:
# Enviamos una solicitud que requiere tanto análisis organizativo como numérico
prompt_sistema_complejo = (
    "El cliente corporativo Stark Industries (SLA contractual de 4 horas) sufrió una "
    "caída masiva de sus servicios que duró un total de 12 horas seguidas. Necesito que delegues "
    "una auditoría financiera completa para determinar la penalización en dólares según su nivel "
    "de suscripción (GOLD) y me entregues un reporte ejecutivo de la situación."
)

run_hub_orchestrator(prompt_sistema_complejo)

[🔥 HUB START] Solicitud Compleja: El cliente corporativo Stark Industries (SLA contractual de 4 horas) sufrió una caída masiva de sus servicios que duró un total de 12 horas seguidas. Necesito que delegues una auditoría financiera completa para determinar la penalización en dólares según su nivel de suscripción (GOLD) y me entregues un reporte ejecutivo de la situación.

🎯 Hub decide delegar usando la herramienta: 'delegate_financial_audit'
🛰️ [Hub Orchestrator] Delegando auditoría financiera a subagente...
🤖 [Spoke Agent] Ejecutando Subagente de Impacto Financiero con contexto aislado...

👑 [RESPUESTA FINAL SINTETIZADA POR EL HUB]:
---

## 📋 REPORTE EJECUTIVO — INCIDENTE STARK INDUSTRIES

---

### 🔴 RESUMEN DE SITUACIÓN

| Parámetro | Detalle |
|---|---|
| **Cliente** | Stark Industries |
| **Nivel de Suscripción** | 🥇 GOLD |
| **SLA Contractual** | 4 horas máximo de caída permitida |
| **Duración Real del Incidente** | 12 horas continuas |
| **Excedente SLA (Breach)** | ⚠️ **8 horas 